#### Inisialisasi

In [2]:
import findspark
findspark.init()

import sys
import io
import math
import mlflow
import mlflow.spark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = SparkSession.builder \
    .appName("KMeans_Pipeline_ASEAN") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("KMeans_Zonasi_Wilayah_ASEAN")


c:\Users\marit\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1779349567293, experiment_id='1', last_update_time=1779349567293, lifecycle_stage='active', name='KMeans_Zonasi_Wilayah_ASEAN', tags={}, trace_location=None, workspace='default'>

#### Konfigurasi

In [3]:
PATH_INPUT = "hdfs://localhost:9000/Project_akhir/data_bersih_asean"

print(f"Membaca data bersih dari HDFS: {PATH_INPUT}")
df = spark.read.parquet(PATH_INPUT)

df = df.cache()
total_data = df.count()
print(f"Sukses Load Data! Total baris data yang siap diklasterisasi: {total_data:,}")

Membaca data bersih dari HDFS: hdfs://localhost:9000/Project_akhir/data_bersih_asean
Sukses Load Data! Total baris data yang siap diklasterisasi: 3,857,065


#### Pencarian K Optimal menggunakan Silhouette Score

In [4]:
best_silhouette = -1
best_k = 2
best_model = None
best_predictions = None
best_wssse = None

print("Memulai Proses Pencarian K Cluster Terbaik (K=2 sampai K=5)")

for k in range(2, 6):
    with mlflow.start_run(run_name=f"KMeans_K_{k}"):
        # ambil model dg kolom vector prediction_features & melatih data 100%
        kmeans = KMeans(
            featuresCol="prediction_features", 
            predictionCol="prediction", 
            seed=42, 
            k=k
        )
        model = kmeans.fit(df)
        
        # Prediksi label cluster (kolom 'prediction')
        predictions_full = model.transform(df)

        # Log likelihood, cost, dan cluster centers untuk analisis lebih lanjut
        wssse = model.summary.trainingCost 
        mlflow.log_metric("training_cost", model.summary.trainingCost)
        mlflow.log_metric("wssse", wssse)
        
        # Evaluasi Silhouette menggunakan 10% sampel 
        df_sample_eval = predictions_full.sample(False, 0.1, seed=42)
        evaluator = ClusteringEvaluator(
            featuresCol="prediction_features", 
            metricName="silhouette",
            distanceMeasure="squaredEuclidean"
        )
        silhouette = evaluator.evaluate(df_sample_eval)
        
        # Logging parameter dan metrik secara otomatis ke dashboard MLflow
        mlflow.log_param("k", k)
        mlflow.log_metric("silhouette_score", silhouette)
        mlflow.log_metric("wssse",            wssse)
        print(f"Iterasi K={k} Selesai | Silhouette Score: {silhouette:.4f}")
        
        # Menyimpan model dengan struktur klaster terbaik
        if silhouette > best_silhouette:
            best_silhouette = silhouette
            best_k = k
            best_model = model
            best_predictions = predictions_full
            best_wssse = wssse

print(f"\n>>> K={best_k} dengan Silhouette Score: {best_silhouette:.4f} dan nilai wssse {wssse:>16.2f} <<<")

Memulai Proses Pencarian K Cluster Terbaik (K=2 sampai K=5)
Iterasi K=2 Selesai | Silhouette Score: 0.6976
🏃 View run KMeans_K_2 at: http://localhost:5000/#/experiments/1/runs/1f27ea6f84704d73be612dea57200fea
🧪 View experiment at: http://localhost:5000/#/experiments/1
Iterasi K=3 Selesai | Silhouette Score: 0.6448
🏃 View run KMeans_K_3 at: http://localhost:5000/#/experiments/1/runs/15769660ce244ed1aa4bea23a98c8625
🧪 View experiment at: http://localhost:5000/#/experiments/1
Iterasi K=4 Selesai | Silhouette Score: 0.8029
🏃 View run KMeans_K_4 at: http://localhost:5000/#/experiments/1/runs/b0bb53e698ee4bd89ba3543fd38cfb58
🧪 View experiment at: http://localhost:5000/#/experiments/1
Iterasi K=5 Selesai | Silhouette Score: 0.8056
🏃 View run KMeans_K_5 at: http://localhost:5000/#/experiments/1/runs/3cac6c0901394174960e1ed416071f4d
🧪 View experiment at: http://localhost:5000/#/experiments/1

>>> K=5 dengan Silhouette Score: 0.8056 dan nilai wssse         27478.68 <<<


#### Penyimpanan ke mlflow

In [7]:
import shutil
import os

LOCAL_TEMP_DIR = "C:/hadoop/tmp/spark_model_kmeans"

if os.path.exists(LOCAL_TEMP_DIR):
    shutil.rmtree(LOCAL_TEMP_DIR)

best_model.save(LOCAL_TEMP_DIR)

# Logging ml flow
with mlflow.start_run(run_name=f"Final_Best_Model_K{best_k}"):
    mlflow.log_param("best_k",            best_k)
    mlflow.log_metric("silhouette_score", best_silhouette)
    mlflow.log_metric("wssse",            best_wssse)
    
    # Menggunakan log_artifacts untuk mengunggah folder model lokal tadi ke server MLflow
    mlflow.log_artifacts(LOCAL_TEMP_DIR, artifact_path="model_kmeans_asean_optimal")

print(f"Sukses! Model K={best_k} berhasil di-log ke MLflow.")
print(f"  Silhouette : {best_silhouette:.4f}")

🏃 View run Final_Best_Model_K5 at: http://localhost:5000/#/experiments/1/runs/38e8d2df8cf54c518d6175b1d0cfa616
🧪 View experiment at: http://localhost:5000/#/experiments/1
Sukses! Model K=5 berhasil di-log ke MLflow.
  Silhouette : 0.8056


#### Simpan hasil

In [8]:
if best_model is not None:
    path_output_full = "hdfs://localhost:9000/Project_akhir/hasil_clustering_asean"
    
    print(f"Menyimpan hasil klasterisasi lengkap ke HDFS: {path_output_full}")
    best_predictions.write.mode("overwrite").parquet(path_output_full)
    print("Data berhasil disimpan!")

Menyimpan hasil klasterisasi lengkap ke HDFS: hdfs://localhost:9000/Project_akhir/hasil_clustering_asean
Data berhasil disimpan!


#### Pengecekan & Validasi

In [9]:
print("Cluster unik yang terbentuk")
best_predictions.select("prediction").distinct().show()

print("Sampel Data Hasil Clustering")
best_predictions.select("Country", "Network", "radio", "prediction").show(10)

print("Distribusi Jumlah Menara per Cluster")
best_predictions.groupBy("prediction").count().orderBy("prediction").show()

Cluster unik yang terbentuk
+----------+
|prediction|
+----------+
|         1|
|         4|
|         0|
|         3|
|         2|
+----------+

Sampel Data Hasil Clustering
+---------+---------+-----+----------+
|  Country|  Network|radio|prediction|
+---------+---------+-----+----------+
|Indonesia|Telkomsel| UMTS|         0|
|Indonesia|Telkomsel| UMTS|         4|
|Indonesia|Telkomsel| UMTS|         1|
|Indonesia|Telkomsel| UMTS|         1|
|Indonesia|Telkomsel|  GSM|         0|
|Indonesia|Telkomsel|  LTE|         1|
|Indonesia|Telkomsel| UMTS|         4|
|Indonesia|Telkomsel| UMTS|         4|
|Indonesia|Telkomsel| UMTS|         4|
|Indonesia|Telkomsel| UMTS|         4|
+---------+---------+-----+----------+
only showing top 10 rows
Distribusi Jumlah Menara per Cluster
+----------+-------+
|prediction|  count|
+----------+-------+
|         0| 923291|
|         1| 195316|
|         2| 291438|
|         3|1500397|
|         4| 946623|
+----------+-------+



#### Profiling

In [ ]:
path_output_viz = "hdfs://localhost:9000/Project_akhir/visualisasi_asean/profiling_cluster"

# 1. PROFILING STATISTIK UTAMA (Titik Tengah Spasial & Rata-rata Karakteristik)
## memetakan lokasi dominan tiap cluster di peta/heatmap.
cluster_stat = best_predictions.groupBy("prediction") \
    .agg(
        F.avg("LAT").alias("avg_lat"),
        F.avg("LON").alias("avg_lon"),
        F.avg("RANGE").alias("avg_range_radius"),
        F.avg("SAM").alias("avg_sample_count"),
        F.count("*").alias("total_tower")
    ).orderBy("prediction")

cluster_stat.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{path_output_viz}/stats_utama")

# 2. PROFILING HIERARKI (Cluster -> Negara -> Operator Dominan)
cluster_hierarchy = best_predictions.groupBy("prediction", "Country", "Network") \
    .count() \
    .orderBy("prediction", "Country", F.desc("count"))

cluster_hierarchy.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{path_output_viz}/Hierarki-Cluster-Lengkap")
# 3. PROFILING DOMINASI TEKNOLOGI JARINGAN (Generasi Radio)
cluster_tech = best_predictions.groupBy("prediction", "generasi") \
    .count() \
    .orderBy("prediction", F.desc("count"))

cluster_tech.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{path_output_viz}/Dominasi-teknologi")

# 4. PROFILING TIPE JANGKAUAN WILAYAH (Urban, Suburban, Rural)
cluster_area = best_predictions.groupBy("prediction", "jangkauan") \
    .count() \
    .orderBy("prediction", F.desc("count"))

cluster_area.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{path_output_viz}/Dominasi-Wilayah")

print(f"Seluruh File Profiling K-Means Berhasil Disimpan di HDFS: {path_output_viz}")

Seluruh File Profiling K-Means Berhasil Disimpan di HDFS: hdfs://localhost:9000/Project_akhir/visualisasi_asean/profiling_cluster


In [3]:
try:
    # 1. Tentukan Path HDFS
    path_input_clustering = "hdfs://localhost:9000/Project_akhir/hasil_clustering_asean"
    path_sample_map = "hdfs://localhost:9000/Project_akhir/visualisasi_asean/profiling_cluster/sample_map_tower"

    print(">>> Membaca hasil klasterisasi K-Means yang sudah ada dari HDFS...")
    df_hasil = spark.read.parquet(path_input_clustering)
    
    # Hitung total data secara dinamis
    total_records = df_hasil.count()
    print(f">>> Sukses memuat data. Total record ditemukan: {total_records:,}")

    # 2. Ambil sampel 10.000 menara secara acak dan seleksi kolom yang dibutuhkan saja
    print(">>> Memulai proses pengambilan sampel 10.000 menara...")
    df_sample = df_hasil.select("LAT", "LON", "prediction", "Country", "Network") \
        .sample(withReplacement=False, fraction=10000/total_records, seed=42)

    # 3. Simpan ke HDFS dalam format CSV untuk dibaca Streamlit
    print(f">>> Menyimpan file sampel ke: {path_sample_map}")
    df_sample.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv(path_sample_map)
    
except Exception as e:
    print(f"[ERROR] Terjadi kendala: {str(e)}")

finally:
    spark.stop()

>>> Membaca hasil klasterisasi K-Means yang sudah ada dari HDFS...
>>> Sukses memuat data. Total record ditemukan: 3,857,065
>>> Memulai proses pengambilan sampel 10.000 menara...
>>> Menyimpan file sampel ke: hdfs://localhost:9000/Project_akhir/visualisasi_asean/profiling_cluster/sample_map_tower
